This notebook is implemented to split the __Gowalla__ and __Tmall__ datasets into training/validation/test sets, the datasets are given by the authors of the [LightGCL](https://openreview.net/forum?id=FKXVK9dyMM) paper on their [Github repository](https://github.com/HKUDS/LightGCL?tab=readme-ov-file). The reason we use these datasets is to ensure the fairness between __DiffRec__ with 3 selected alternative methods _LightGCN_, _SimGCL_ and _LightGCL_ (which are already tested on these 2 datasets, their results are shown in the __Table 1__ in the LightGCL paper).

# Importing packages

In [50]:
import pickle
import os
import pandas as pd
import numpy as np
from scipy.sparse import coo_matrix, csr_matrix

# Fixing seed for reproducibility

In [51]:
seed = 24
np.random.seed(seed)

# Loading data

In [52]:
# Helper function to read pickle files
def read_pkl(file_path):
    with open(file_path, 'rb') as file:
        data = pickle.load(file)
    
    return data

In [53]:
gowalla_mat = read_pkl("gowalla/trnMat.pkl")
tmall_mat = read_pkl("tmall/trnMat.pkl")

C:\Users\Sn\AppData\Local\Temp\ipykernel_20804\374016073.py:4: DeprecationWarning: Please import `coo_matrix` from the `scipy.sparse` namespace; the `scipy.sparse.coo` namespace is deprecated and will be removed in SciPy 2.0.0.
  data = pickle.load(file)
C:\Users\Sn\AppData\Local\Temp\ipykernel_20804\374016073.py:4: DeprecationWarning: numpy.core._multiarray_umath is deprecated and has been renamed to numpy._core._multiarray_umath. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core._multiarray_umath._reconstruct.
  data = pickle.load(file)
C:\Users\Sn\AppData\Local\Temp\ipykernel_20804\374016073.py:4: VisibleDeprecationWarning: dtype(

In [54]:
gowalla_mat

<COOrdinate sparse matrix of dtype 'float64'
	with 1172425 stored elements and shape (50821, 57440)>

In [55]:
tmall_mat

<COOrdinate sparse matrix of dtype 'float64'
	with 2357450 stored elements and shape (47939, 41390)>

Since the `gowalla_mat` and `tmall_mat` now have the same users, items and the number of interactions as shown the section 4.1.1 of the __LightGCL__ paper, we will only work on the _"trnMat.pkl"_, ignoring the _"tstMat.pkl"_ of the two datasets.

# Splitting data

We split each dataset into training, validation and testing sets with a ratio of 7:2:1

In [56]:

def train_val_test_split(data_mat: coo_matrix, val_ratio=0.2, test_ratio=0.1, seed=42):
    """
    Perform train-validation-test splits on given COO matrix
    """
    # Convert to DataFrame
    df = pd.DataFrame({
        "user": data_mat.row,
        "item": data_mat.col,
        "value": data_mat.data
    })

    # Shuffle
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    
    # Rank elements within each user
    df["rank"] = df.groupby("user").cumcount()

    # Count interactions per user
    df["count"] = df.groupby("user")["item"].transform("count")
    
    # Compute split thresholds
    test_cut = (df["count"] * test_ratio).clip(lower=1)
    val_cut = (df["count"] * (test_ratio + val_ratio)).clip(lower=2)
    
    # Assign splits, skip for users who have <= 2 records 
    df["is_test"] = (df["count"] > 2) & (df["rank"] < test_cut)

    df["is_val"] = (df["count"] > 2) & \
                (~df["is_test"]) & \
                (df["rank"] < val_cut)
        
    # Split DataFrames
    test_df = df[df["is_test"]]
    val_df = df[df["is_val"]]
    train_df = df[~(df["is_test"] | df["is_val"])]
    
    # Convert to CSR
    train_csr = coo_matrix(
        (train_df["value"], (train_df["user"], train_df["item"])),
        shape=data_mat.shape
    ).tocsr()
    
    val_csr = coo_matrix(
        (val_df["value"], (val_df["user"], val_df["item"])),
        shape=data_mat.shape
    ).tocsr()
    
    test_csr = coo_matrix(
        (test_df["value"], (test_df["user"], test_df["item"])),
        shape=data_mat.shape
    ).tocsr()
    
    return train_csr, val_csr, test_csr

In [57]:
gowalla_train, gowalla_val, gowalla_test = train_val_test_split(gowalla_mat, val_ratio=0.2, test_ratio=0.1)

In [58]:
tmall_train, tmall_val, tmall_test = train_val_test_split(tmall_mat, val_ratio=0.2, test_ratio=0.1)

In [59]:
print("Gowalla")
print("Type  - Users - Items - Interactions")
print(f"Train\t{gowalla_train.shape[0]}\t{gowalla_train.shape[1]}\t{gowalla_train.data.shape[0]}")
print(f"Valid\t{gowalla_val.shape[0]}\t{gowalla_val.shape[1]}\t{gowalla_val.data.shape[0]}")
print(f"Test \t{gowalla_test.shape[0]}\t{gowalla_val.shape[1]}\t{gowalla_test.data.shape[0]}")

Gowalla
Type  - Users - Items - Interactions
Train	50821	57440	792344
Valid	50821	57440	241211
Test 	50821	57440	138870


In [60]:
print("Tmall")
print("Type  - Users - Items - Interactions")
print(f"Train\t{tmall_train.shape[0]}\t{tmall_train.shape[1]}\t{tmall_train.data.shape[0]}")
print(f"Valid\t{tmall_val.shape[0]}\t{tmall_val.shape[1]}\t{tmall_val.data.shape[0]}")
print(f"Test \t{tmall_test.shape[0]}\t{tmall_test.shape[1]}\t{tmall_test.data.shape[0]}")

Tmall
Type  - Users - Items - Interactions
Train	47939	41390	1623850
Valid	47939	41390	476268
Test 	47939	41390	257332


# Save to files

In [61]:
def save_sparse(sparse_mat, filename, dir_name=""):
    # Save to file for DiffRec input
    # save_npz(f"{path}/{filename}.npz", sparse_mat)

    rows, cols = sparse_mat.nonzero()

    data_to_save = np.stack([rows, cols], axis=1)

    np.save(f"{dir_name}/{filename}", data_to_save)

In [62]:
save_sparse(gowalla_train, 'train_list.npy', 'gowalla')
save_sparse(gowalla_val, 'valid_list', 'gowalla')
save_sparse(gowalla_test, 'test_list', 'gowalla')

save_sparse(tmall_train, 'train_list.npy', 'tmall')
save_sparse(tmall_val, 'valid_list', 'tmall')
save_sparse(tmall_test, 'test_list', 'tmall')